# 🐙 Lab 10: Run ArgoCD on Minikube Using ArgoCD Helm Chart & Access UI

🕒 **Estimated Time:** 25–35 Minutes  
🎯 **Focus Areas:** GitOps, ArgoCD, Helm 3, Minikube, Port-Forwarding, Secret Management, Application Manifests

---

## 🧾 Lab Overview
**ArgoCD** is a declarative, GitOps continuous delivery tool for Kubernetes. In this hands-on lab, you will deploy ArgoCD into a **Minikube** cluster using the official **Helm Chart**, retrieve the administrative credentials, access the web console, and deploy an automated GitOps application.

```
                      +------------------------------------------+
                      |             Minikube Cluster             |
                      |                                          |
                      |   Namespace: 'argocd'                    |
                      |  +------------------------------------+  |
  Browser / User      |  | ArgoCD Server (Port 80/443)        |  |
  http://localhost:8080 ----> [Service: argocd-server]        |  |
        |             |  +------------------------------------+  |
        |             |  +------------------------------------+  |
        v             |  | ArgoCD Repo Server                 |  |
  [Git Repository]    |  | ArgoCD Application Controller      |  |
  (e.g., Guestbook) <------- Syncs desired state to cluster   |  |
                      |  +------------------------------------+  |
                      +------------------------------------------+
```

---

## 🎯 Objectives
- 🚀 Verify Minikube cluster and client tooling
- 📦 Add and update the official `argo` Helm repository
- ⚙️ Configure custom values for local developer access (`--insecure` mode)
- 🛠️ Install ArgoCD using Helm 3
- 🔐 Extract and decode the initial admin password from Kubernetes secrets
- 🌐 Expose the ArgoCD web server and open the web UI at `http://localhost:8080`
- 📋 Deploy a sample GitOps application (`guestbook`)
- 🧹 Tear down and clean up resources safely


---
## ☘️ Step 1: Verify Kubernetes & Helm Environment
Check that your Minikube cluster and Helm client are functional.


In [ ]:
!kubectl cluster-info


In [ ]:
!helm version


---
## ☘️ Step 2: Create the `argocd` Namespace
Create a dedicated namespace to isolate all ArgoCD controllers and services.


In [ ]:
!kubectl create namespace argocd


In [ ]:
!kubectl get namespaces


---
## ☘️ Step 3: Add the Argo Helm Repository
Add the official Argo project repository and fetch the latest chart indexes.


In [ ]:
!helm repo add argo https://argoproj.github.io/argo-helm
!helm repo update


In [ ]:
!helm search repo argo/argo-cd


---
## ☘️ Step 4: Inspect Custom Values (`argocd-values.yaml`)
To make local access effortless without browser HTTPS certificate errors, we set `server.extraArgs: ["--insecure"]`.


In [ ]:
with open('argocd-values.yaml', 'r') as f:
    print(f.read())


---
## ☘️ Step 5: Install ArgoCD with Helm
Install the chart into the `argocd` namespace using our custom values file.


In [ ]:
!helm install argocd argo/argo-cd -n argocd -f argocd-values.yaml


In [ ]:
!helm list -n argocd


---
## ☘️ Step 6: Verify Pod Health in `argocd` Namespace
Check that all ArgoCD pods (server, repo-server, controller, redis, dex) are initializing.


In [ ]:
!kubectl get pods -n argocd


> ⏳ Note: Pulling container images may take 1–2 minutes on first run. Run the cell below periodically until all pods are `Running`.


In [ ]:
!kubectl get pods -n argocd -o wide


---
## ☘️ Step 7: Retrieve and Decode the Admin Password
ArgoCD creates a Kubernetes secret named `argocd-initial-admin-secret`. Let's decode it using Python:


In [ ]:
import subprocess
import base64

try:
    cmd = ["kubectl", "-n", "argocd", "get", "secret", "argocd-initial-admin-secret", "-o", "jsonpath={.data.password}"]
    encoded_pw = subprocess.check_output(cmd).decode().strip()
    decoded_pw = base64.b64decode(encoded_pw).decode('utf-8')
    print("=" * 50)
    print("🔑 ArgoCD Login Credentials:")
    print("Username: admin")
    print(f"Password: {decoded_pw}")
    print("=" * 50)
except Exception as e:
    print("Error retrieving password (check if secret is ready):", e)


---
## ☘️ Step 8: Port-Forward ArgoCD Server
To access the web console on your local browser, port-forward service `argocd-server` to port `8080`.

> **Note:** Run the command below in an external terminal window or PowerShell so it runs in the background:
> ```bash
> kubectl port-forward svc/argocd-server -n argocd 8080:80
> ```
> Then open your browser at: **[http://localhost:8080](http://localhost:8080)**


In [ ]:
# Inspect the argocd-server service definition
!kubectl get svc argocd-server -n argocd


---
## ☘️ Step 9: Deploy a Sample GitOps Application
Let's inspect and apply an ArgoCD `Application` custom resource that synchronizes the classic **Guestbook** application from Git.


In [ ]:
with open('guestbook-app.yaml', 'r') as f:
    print(f.read())


In [ ]:
!kubectl apply -f guestbook-app.yaml


In [ ]:
# Inspect the newly created application CRD
!kubectl get applications -n argocd


---
## ☘️ Step 10: Verify Deployed Application Pods
Once ArgoCD syncs the application, the guestbook pods and service will appear in the `default` namespace.


In [ ]:
!kubectl get all -l app.kubernetes.io/instance=guestbook


---
## ☘️ Step 11: Cleanup Resources
When finished with the lab, clean up all deployed applications and Helm releases.


In [ ]:
# 1. Remove the sample application
!kubectl delete -f guestbook-app.yaml

# 2. Uninstall the ArgoCD Helm release
!helm uninstall argocd -n argocd

# 3. Delete the argocd namespace
!kubectl delete namespace argocd


In [ ]:
# Verify cleanup
!kubectl get namespaces


---
## ✅ Conclusion
Congratulations! You have successfully:
- Installed and configured **ArgoCD** on **Minikube** using **Helm 3**
- Extracted and decoded initial cluster credentials
- Connected to the ArgoCD web UI console
- Synchronized a declarative GitOps application from a Git repository
- Managed resource lifecycles using Helm and kubectl

🎉 **END OF LAB 10** 🎉
